[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kewel-corp/book-vol2/blob/main/notebooks/ch07_code.ipynb)

# 第7章　ベースラインとアブレーション：何が性能を上げているかを確かめる

**『医療診断支援AI開発　第2巻　応用編 ― 本格実装から臨床導入まで（第2巻　応用編）』のコード**

本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。

- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。
- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。
- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**

リポジトリ: https://github.com/kewel-corp/book-vol2

## 計算負担の大きい実験のアブレーション：低コストのプロキシと、患者単位のブートストラップ

In [ ]:
import numpy as np
def bootstrap_ci(per_case_dice, n=2000, alpha=0.05, seed=0):
    d = np.asarray(per_case_dice)               # 症例ごとのDice（1患者1症例を仮定）
    rng = np.random.default_rng(seed)           # 再標本化も種を固定して再現できるようにする
    boot = [np.mean(rng.choice(d, len(d), replace=True)) for _ in range(n)]
    return np.mean(d), np.quantile(boot, [alpha/2, 1-alpha/2])
# 例：mean 0.49, 95%CI [0.41, 0.56]
# ※1患者に複数症例があるなら、症例ではなく患者を復元抽出し、その患者の症例をまとめて入れる。
# ※2つのモデルを比べるときは、それぞれのCIの重なりを見てはいけない。同じ症例で測る以上、
#   比較は「対応のある差」になる。症例ごとの差 d_i = A_i - B_i を作り、その差を患者単位で
#   再標本化して差のCIを出す。各モデルのCIが重なっていても、差のCIが0をまたがないことは普通に起きる。